# 03 · Zero-inflation (ZINB) vs plain NegBin

Compare **Bayes-NegBin** and **Bayes-ZINB** on one high-zero and one low-zero
series from `long.parquet`.

**ZINB generative story**

$$
\begin{aligned}
\mathrm{logit}(\psi_t) &= \gamma_0 + \gamma_{\mathrm{dow}[d(t)]} + \gamma_s\,\mathrm{snap}_t + \gamma_e\,\mathrm{is\_event}_t \\
\log \mu_t &= \beta_0 + \beta_{\mathrm{dow}[d(t)]} + \beta_p\,\tilde{p}_t \\
y_t &\sim \mathrm{ZINB}(\psi_t,\ \mu_t,\ \alpha)
\end{aligned}
$$

Occurrence (SNAP / events) drives **ψ**; size when demand happens is **μ**.
Point forecast: $\mathbb{E}[y_t] = (1-\psi_t)\,\mu_t$.

```bash
uv sync --group bayesian
make prep
```

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import os

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from m5.config import REPO_ROOT, SETTINGS, set_global_seed
from m5.evaluation import compute_components, wrmsse
from m5.models.bayesian import (
    extract_posterior,
    fit_bayes_negbin,
    fit_bayes_zinb,
    pick_intermittency_examples,
    posterior_mean_forecast_negbin,
    posterior_mean_forecast_zinb,
    posterior_zero_prob_zinb,
    series_zero_rate,
)
from m5.plots import configure_style

_pytensor_dir = REPO_ROOT / ".pytensor"
_pytensor_dir.mkdir(exist_ok=True)
os.environ.setdefault("PYTENSOR_FLAGS", f"compiledir={_pytensor_dir}")

configure_style()
set_global_seed()

HORIZON = SETTINGS.horizon
LAST_N_DAYS = min(SETTINGS.last_n_days, 200)
MCMC_DRAWS = 500
MCMC_TUNE = 500
LONG_PARQUET = SETTINGS.processed_dir / "long.parquet"

## 1 · Pick high-zero vs low-zero hero series

In [ ]:
if not LONG_PARQUET.exists():
    raise FileNotFoundError(f"Missing {LONG_PARQUET} — run `make prep` first.")

long = pd.read_parquet(LONG_PARQUET)
long["ds"] = pd.to_datetime(long["ds"])
window = long.sort_values(["unique_id", "ds"]).groupby("unique_id", observed=True).tail(LAST_N_DAYS + HORIZON)

high_id, low_id = pick_intermittency_examples(window)
rates = series_zero_rate(window).loc[[high_id, low_id]]
print(rates.rename("zero_rate"))

examples = {}
for label, uid in {"high_zero": high_id, "low_zero": low_id}.items():
    df = window.loc[window["unique_id"] == uid].sort_values("ds")
    examples[label] = {
        "uid": uid,
        "train": df.iloc[:-HORIZON].copy(),
        "test": df.iloc[-HORIZON:].copy(),
    }

## 2 · Fit NegBin and ZINB on the high-zero series

In [ ]:
train = examples["high_zero"]["train"]
test = examples["high_zero"]["test"]
uid = examples["high_zero"]["uid"]

neg_idata = fit_bayes_negbin(train, draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=2, progressbar=True)
zinb_idata = fit_bayes_zinb(train, draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=2, progressbar=True)

neg_post = extract_posterior(neg_idata)
zinb_post = extract_posterior(zinb_idata)

y_neg = posterior_mean_forecast_negbin(neg_post, test)
y_zinb = posterior_mean_forecast_zinb(zinb_post, test)
p_zero = posterior_zero_prob_zinb(zinb_post, test)

print(f"series: {uid}")
print(f"train zero-rate: {(train['y'] == 0).mean():.1%}")
print(f"hold-out zero-rate: {(test['y'] == 0).mean():.1%}")
print(f"ZINB mean P(y=0) on hold-out: {p_zero.mean():.1%}")

## 3 · Zero-rate calibration (hold-out)

In [ ]:
cal = pd.DataFrame(
    {
        "ds": test["ds"],
        "actual_zero": (test["y"].to_numpy() == 0).astype(float),
        "zinb_p_zero": p_zero,
    }
)
cal["neg_p_zero"] = np.exp(-np.maximum(y_neg, 1e-9))  # Poisson/NegBin rough proxy: exp(-mean)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(cal["ds"], cal["actual_zero"], "o", ms=4, label="actual y=0", color="black")
ax.plot(cal["ds"], cal["zinb_p_zero"], "-", lw=2, label="ZINB P(y=0)")
ax.plot(cal["ds"], cal["neg_p_zero"], "--", lw=1.5, label="NegBin proxy P(y=0)")
ax.set_title(f"Zero calibration — high-zero series {uid}")
ax.set_ylabel("probability")
ax.legend(loc="upper left")
plt.tight_layout()

## 4 · Point forecast comparison (both hero series)

In [ ]:
rows = []
for label in ("high_zero", "low_zero"):
    train_i = examples[label]["train"]
    test_i = examples[label]["test"]
    uid_i = examples[label]["uid"]

    neg_i = extract_posterior(fit_bayes_negbin(train_i, draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=2))
    zinb_i = extract_posterior(fit_bayes_zinb(train_i, draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=2))

    y_neg_i = posterior_mean_forecast_negbin(neg_i, test_i)
    y_zinb_i = posterior_mean_forecast_zinb(zinb_i, test_i)

    comp = compute_components(train_i)
    truth = test_i[["unique_id", "ds", "y"]]

    for model, y_hat in {"Bayes-NegBin": y_neg_i, "Bayes-ZINB": y_zinb_i}.items():
        fcst = test_i[["unique_id", "ds"]].copy()
        fcst["y_hat"] = y_hat
        rows.append(
            {
                "series": label,
                "unique_id": uid_i,
                "model": model,
                "MAE": float(np.abs(y_hat - test_i["y"].to_numpy()).mean()),
                "WRMSSE": wrmsse(truth, fcst, comp, forecast_col="y_hat"),
                "train_zero_rate": float((train_i["y"] == 0).mean()),
            }
        )

summary = pd.DataFrame(rows).set_index(["series", "model"]).sort_values(["series", "WRMSSE"])
summary.style.format({"MAE": "{:.3f}", "WRMSSE": "{:.4f}", "train_zero_rate": "{:.1%}"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, label in zip(axes, ("high_zero", "low_zero"), strict=True):
    test_i = examples[label]["test"]
    train_i = examples[label]["train"]
    neg_i = extract_posterior(fit_bayes_negbin(train_i, draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=2))
    zinb_i = extract_posterior(fit_bayes_zinb(train_i, draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=2))
    y_neg_i = posterior_mean_forecast_negbin(neg_i, test_i)
    y_zinb_i = posterior_mean_forecast_zinb(zinb_i, test_i)

    hist = train_i.tail(56)
    ax.plot(hist["ds"], hist["y"], color="0.5", lw=1, label="train")
    ax.plot(test_i["ds"], test_i["y"], "o", ms=3, color="black", label="actual")
    ax.plot(test_i["ds"], y_neg_i, "-", label="NegBin")
    ax.plot(test_i["ds"], y_zinb_i, "-", label="ZINB")
    ax.set_title(f"{label} — {examples[label]['uid']}")
    ax.legend(loc="upper left", fontsize=8)

plt.tight_layout()

## 5 · Batch CV with `likelihood='zinb'`

For panel evaluation against stats / LGBM, use notebook `02_cv_evaluation.ipynb`
and pass `likelihood='zinb'` to `bayesian_cv`:

```python
from m5.cv import bayesian_cv

cv_zinb = bayesian_cv(eval_long, likelihood="zinb", n_series=15, quiet=True)
```

**Next step for production-scale intermittency:** hierarchical partial pooling on
`gamma` and `beta` intercepts by `dept_id` / `store_id` (see partial-pooling
section in `01_hierarchical_negbin.ipynb`), routed by ADI/CV² series class.